Hybrid Siamese Network 

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, Flatten, GlobalAveragePooling2D, Concatenate, Dropout, Subtract, Lambda
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

# Set image size for InceptionV3
image_size = (299, 299)

# Load handcrafted features
features_df = pd.read_csv("../Dataset/Features/signature_features.csv")
features_dict = {row["filename"]: row.iloc[2:].values for _, row in features_df.iterrows()}  # Map filenames to feature vectors

# Function to load and preprocess images
def load_image(image_path):
    img = load_img(image_path, target_size=image_size)
    img = img_to_array(img)
    img = tf.keras.applications.inception_v3.preprocess_input(img)
    return img

# Function to create positive and negative pairs
def create_pairs(data_dir):
    pairs, labels = [], []

    genuine_paths = {file: os.path.join(data_dir, "Genuine", file) for file in os.listdir(os.path.join(data_dir, "Genuine")) if file.endswith(".jpg")}
    forged_paths = {file: os.path.join(data_dir, "Forged", file) for file in os.listdir(os.path.join(data_dir, "Forged")) if file.endswith(".jpg")}

    # Extract person-wise groups
    person_signatures = {}
    for filename in genuine_paths.keys():
        person_id = "_".join(filename.split("_")[1:3])  # Extract DX_XX as person ID
        if person_id not in person_signatures:
            person_signatures[person_id] = {"genuine": [], "forged": []}
        person_signatures[person_id]["genuine"].append(filename)

    for filename in forged_paths.keys():
        person_id = "_".join(filename.split("_")[1:3])
        if person_id in person_signatures:
            person_signatures[person_id]["forged"].append(filename)

    # Creating pairs
    for person_id, files in person_signatures.items():
        genuine_images = files["genuine"]
        forged_images = files["forged"]

        # Positive Pairs (Genuine, Genuine)
        for i in range(len(genuine_images) - 1):
            pairs.append((genuine_paths[genuine_images[i]], genuine_paths[genuine_images[i + 1]]))
            labels.append(1)

        # Negative Pairs (Genuine, Forged)
        for genuine in genuine_images:
            for forged in forged_images:
                pairs.append((genuine_paths[genuine], forged_paths[forged]))
                labels.append(0)

    return pairs, labels

# Create training and testing pairs
dataset_path = "../Dataset/Processing/AugmentedDataset"
pairs, labels = create_pairs(dataset_path)
train_pairs, test_pairs, y_train, y_test = train_test_split(pairs, labels, test_size=0.2, random_state=42, stratify=labels)

# Load images and handcrafted features into arrays
def prepare_data(pairs):
    img1, img2, feats1, feats2 = [], [], [], []
    
    for path1, path2 in pairs:
        img1.append(load_image(path1))
        img2.append(load_image(path2))
        
        filename1 = os.path.basename(path1)
        filename2 = os.path.basename(path2)
        
        feats1.append(features_dict[filename1])
        feats2.append(features_dict[filename2])

    return np.array(img1), np.array(img2), np.array(feats1), np.array(feats2)

X_train_img1, X_train_img2, X_train_feat1, X_train_feat2 = prepare_data(train_pairs)
X_test_img1, X_test_img2, X_test_feat1, X_test_feat2 = prepare_data(test_pairs)

# Reshape handcrafted features for LSTM input
X_train_feat1 = X_train_feat1.reshape(X_train_feat1.shape[0], X_train_feat1.shape[1], 1)
X_train_feat2 = X_train_feat2.reshape(X_train_feat2.shape[0], X_train_feat2.shape[1], 1)
X_test_feat1 = X_test_feat1.reshape(X_test_feat1.shape[0], X_test_feat1.shape[1], 1)
X_test_feat2 = X_test_feat2.reshape(X_test_feat2.shape[0], X_test_feat2.shape[1], 1)

# Define CNN model (shared weights)
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)
cnn_model = Model(inputs=inception_base.input, outputs=cnn_output)

# Define LSTM model (shared weights)
lstm_input = Input(shape=(X_train_feat1.shape[1], 1))
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)
lstm_model = Model(inputs=lstm_input, outputs=lstm_layer)

# Define input layers
img_input1 = Input(shape=(299, 299, 3))
img_input2 = Input(shape=(299, 299, 3))
feat_input1 = Input(shape=(X_train_feat1.shape[1], 1))
feat_input2 = Input(shape=(X_train_feat2.shape[1], 1))

# Feature extraction
cnn_feat1 = cnn_model(img_input1)
cnn_feat2 = cnn_model(img_input2)
lstm_feat1 = lstm_model(feat_input1)
lstm_feat2 = lstm_model(feat_input2)

# Compute absolute difference
cnn_diff = Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([cnn_feat1, cnn_feat2])
lstm_diff = Lambda(lambda tensors: tf.abs(tensors[0] - tensors[1]))([lstm_feat1, lstm_feat2])

# Concatenate differences
merged = Concatenate()([cnn_diff, lstm_diff])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define Siamese model
model = Model(inputs=[img_input1, img_input2, feat_input1, feat_input2], outputs=output_layer)

# Compile model
model.compile(optimizer=Adam(learning_rate=0.0001), loss='binary_crossentropy', metrics=['accuracy'])

# Train model
model.fit(
    [X_train_img1, X_train_img2, X_train_feat1, X_train_feat2], y_train,
    validation_data=([X_test_img1, X_test_img2, X_test_feat1, X_test_feat2], y_test),
    epochs=25, batch_size=32)

# Save the trained model
model.save("../Model/signature_forgery_siamese_model.h5")

print("Model training completed and saved!")
